# `preprocess_and_split.py` — Data Cleaning & Train/Val/Test Split

## Purpose
Processes the machine-translated English reviews from `data/raw/full_data_en.csv` and produces clean, stratified train/val/test splits in `data/splits/`.

## Processing Pipeline
```
1. Load translated CSV
2. Deep text cleaning:
   a. Strip HTML tags & entities (e.g. &amp; → &)
   b. Remove URLs and emails
   c. Normalise Vietnamese translation artifacts (filler phrases, repeated punctuation)
   d. Collapse whitespace
3. Drop rows with no text or all-NaN aspect labels
4. Build multi-label stratification key per row
5. Two-phase stratified split: 70% train / 15% val / 15% test
   — Rare aspect-classes guaranteed minimum samples in val & test
6. Analyse and document class imbalance
```

## Output Files
| File | Description |
|------|-------------|
| `data/splits/train.csv` | Clean training set (70%) |
| `data/splits/val.csv` | Clean validation set (15%) |
| `data/splits/test.csv` | Clean test set (15%) |
| `data/splits/class_distribution.json` | Per-aspect class counts for each split |
| `class_imbalance.md` | Markdown report of imbalance analysis |

In [1]:
print("\u23f3 Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
import re, unicodedata, html
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
import numpy as np
from sklearn.model_selection import train_test_split
import os, json, logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
log = logging.getLogger(__name__)

print(f"\U0001f552 Done in {time.time() - _start_time:.2f}s")
print("\u2705 Completed: Loading dependencies and libraries.")

⏳ Starting: Loading dependencies and libraries...
🕒 Done in 1.48s
✅ Completed: Loading dependencies and libraries.


## Configuration

In [2]:
print("\u23f3 Starting: Loading configuration...")
import time
_start_time = time.time()
import os
ML_RESEARCH = os.path.dirname(os.path.abspath(''))
os.chdir(ML_RESEARCH)

DATA_PATH      = os.path.join(ML_RESEARCH, 'data', 'raw', 'full_data_en.csv')
OUTPUT_DIR     = os.path.join(ML_RESEARCH, 'data', 'splits')
TRAIN_SPLIT    = 0.70
VAL_SPLIT      = 0.15
TEST_SPLIT     = 0.15
RANDOM_SEED    = 42
TEXT_COLUMNS   = ['data']
ASPECT_COLUMNS = ['stayingpower', 'texture', 'smell', 'price', 'colour', 'shipping', 'packing']

# Rare-class guarantee: each rare class gets at least this many samples in val AND test
MIN_EVAL_SAMPLES = 8

# Vietnamese translation-artifact patterns (common filler left by vi->en MT)
TRANSLATION_ARTIFACTS = [
    r'\bthe product is\b',
    r'\bthe goods\b',
    r'\bgoods received\b',
    r'\bpackaging is\b',
    r'\bthe seller\b(?= (is|sent|ships))',
    r'\border received\b',
    r'\bfast delivery\b\.?\s*$',
    r'\bwill buy again\b\.?\s*$',
    r'\bgood product\b\.?\s*$',
    r'[\u200b-\u200f\u202a-\u202e\ufeff]',
]

# Pre-compiled regex patterns for speed
_ARTIFACT_RE    = [re.compile(p, re.IGNORECASE) for p in TRANSLATION_ARTIFACTS]
_HTML_TAG_RE    = re.compile(r'<[^>]+'+'>')
_HTML_ENTITY_RE = re.compile(r'&(?:#\d+|#x[\da-fA-F]+|[a-zA-Z]+);')
_URL_RE         = re.compile(r'https?://\S+|www\.\S+|ftp://\S+', re.IGNORECASE)
_EMAIL_RE       = re.compile(r'[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}', re.IGNORECASE)

print(f"\U0001f552 Done in {time.time() - _start_time:.2f}s")
print("\u2705 Completed: Loading configuration.")


⏳ Starting: Loading configuration...
🕒 Done in 0.00s
✅ Completed: Loading configuration.


## Text Cleaning Functions

These functions form a **pipeline** applied in order to each review text. They handle the common noise patterns seen in machine-translated Vietnamese reviews.

In [3]:
print("\u23f3 Starting: Defining text cleaning functions...")
import time
_start_time = time.time()

def remove_html(text: str) -> str:
    """Decode HTML entities (&amp; -> &, &#39; -> ') and strip remaining <tags>."""
    text = html.unescape(text)
    text = _HTML_ENTITY_RE.sub(' ', text)
    text = _HTML_TAG_RE.sub(' ', text)
    return text


def remove_urls_and_emails(text: str) -> str:
    """Replace URLs and e-mail addresses with a space."""
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    return text


def fix_translation_artifacts(text: str) -> str:
    """
    Normalise patterns left by the vi->en machine translation pipeline:
    filler phrases, excessive punctuation, and invisible Unicode characters.
    """
    text = re.sub(r'\.{3,}', '\u2026', text)
    text = re.sub(r'!{2,}',   '!', text)
    text = re.sub(r'\?{2,}',  '?', text)
    text = re.sub(r'[\u200b-\u200f\u202a-\u202e\ufeff]', '', text)
    for pat in _ARTIFACT_RE:
        text = pat.sub(' ', text)
    return text


def normalise_whitespace(text: str) -> str:
    """Collapse tabs, newlines, and multiple spaces into a single space."""
    text = re.sub(r'[\t\r\n]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


def clean_text(text) -> str:
    """Master cleaning pipeline: validate -> NFC -> HTML -> URLs -> artifacts -> whitespace."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFC', text)
    text = remove_html(text)
    text = remove_urls_and_emails(text)
    text = fix_translation_artifacts(text)
    text = normalise_whitespace(text)
    return text


# -- Pipeline wrapper functions ------------------------------------------

def load_and_explore_data() -> pd.DataFrame:
    """Load translated CSV and log basic stats."""
    log.info('Loading data from %s ...', DATA_PATH)
    df = pd.read_csv(DATA_PATH)
    log.info('Loaded %d rows x %d columns', len(df), len(df.columns))
    log.info('Columns: %s', df.columns.tolist())
    log.info('Missing values:\n%s', df.isnull().sum().to_string())
    return df


def apply_cleaning_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """Apply clean_text to all TEXT_COLUMNS, log how many rows become empty."""
    log.info('Applying cleaning pipeline to: %s', TEXT_COLUMNS)
    for col in TEXT_COLUMNS:
        if col not in df.columns:
            log.warning("Column '%s' not found, skipping.", col)
            continue
        before_empty = int(df[col].isna().sum() + (df[col] == '').sum())
        df[col] = df[col].progress_apply(clean_text)
        df[col] = df[col].replace('', np.nan)
        after_empty  = int(df[col].isna().sum())
        log.info('  %-15s  newly emptied: %4d  |  total missing: %4d',
                 col, after_empty - before_empty, after_empty)
    return df


def log_cleaning_examples(df_raw: pd.DataFrame, df_clean: pd.DataFrame, col: str = 'data', n: int = 5):
    """Print before/after pairs for a visual sanity check."""
    log.info('=== Cleaning examples (column: %s) ===', col)
    if col not in df_raw.columns:
        return
    sample = df_raw[col].dropna().head(n)
    for idx, raw_val in sample.items():
        clean_val = df_clean.loc[idx, col] if idx in df_clean.index else 'N/A'
        print(f'\n  [BEFORE] {str(raw_val)[:150]}')
        print(f'  [AFTER ] {str(clean_val)[:150]}')

print(f"\U0001f552 Done in {time.time() - _start_time:.2f}s")
print("\u2705 Completed: Defining text cleaning functions.")

⏳ Starting: Defining text cleaning functions...
🕒 Done in 0.00s
✅ Completed: Defining text cleaning functions.


## Stratification Key

Each row gets a composite key representing its full label pattern across all aspects (e.g. `colour_positive|price_negative|smell_neutral`). This ensures the split is stratified over *combinations* of labels, not just individual ones.

Patterns that appear only once are grouped as `'other_rare_combination'` so they don't crash `train_test_split`.

In [4]:
print("\u23f3 Starting: Defining function create_multi_label_target...")
import time
_start_time = time.time()

def create_multi_label_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a composite stratification key per row using all aspect labels.
    Format: 'aspect_label|aspect_label|...' (sorted, only labelled aspects included).
    Singleton patterns are grouped as 'other_rare_combination'.
    """
    log.info('Building stratification key ...')

    def combine_labels(row):
        parts = [
            f"{col}_{row[col]}"
            for col in ASPECT_COLUMNS
            if col in df.columns and pd.notna(row[col]) and row[col] != ''
        ]
        return '|'.join(sorted(parts)) if parts else 'no_labels'

    df['_stratify_key'] = df.apply(combine_labels, axis=1)

    key_counts = df['_stratify_key'].value_counts()
    rare_keys  = key_counts[key_counts < 2].index
    log.info('Rare stratification keys (n=1): %d — grouped as other_rare_combination', len(rare_keys))
    df['_stratify_key'] = df['_stratify_key'].apply(
        lambda x: 'other_rare_combination' if x in rare_keys else x
    )
    return df

print(f"\U0001f552 Done in {time.time() - _start_time:.2f}s")
print("\u2705 Completed: Defining function create_multi_label_target.")

⏳ Starting: Defining function create_multi_label_target...
🕒 Done in 0.00s
✅ Completed: Defining function create_multi_label_target.


## Stratified Splitting

**Two-phase split** designed to guarantee rare aspect-classes appear in val and test:

- **Phase 1 — Rare class reservation**: Any aspect-class with fewer than `rare_threshold` total samples would end up with fewer than `MIN_EVAL_SAMPLES` in val/test under a plain 15% slice. These rows are extracted first and allocated 30% to val, 30% to test, 40% to train — intentionally over-represented in eval sets so performance on rare classes (e.g. `price-negative`) can actually be measured.
- **Phase 2 — Majority split**: The remaining rows are split normally (70/15/15) with `stratify` over the composite key.
- **Merge**: The two halves are concatenated and shuffled.

In [5]:
print("\u23f3 Starting: Defining function perform_stratified_split...")
from tqdm.auto import tqdm
import time
_start_time = time.time()

def perform_stratified_split(df: pd.DataFrame):
    """
    Two-phase stratified split with rare-class minimum guarantee.

    Phase 1: Rows belonging to any aspect-class too rare for a 15% slice to yield
             MIN_EVAL_SAMPLES are extracted and split 40/30/30 (train/val/test),
             intentionally over-representing rare classes in eval sets.
    Phase 2: Remaining majority rows get a normal stratified 70/15/15 split.
    Result:  val and test each contain >= MIN_EVAL_SAMPLES of every rare class.
    """
    rare_threshold = int(MIN_EVAL_SAMPLES / TEST_SPLIT)
    log.info('Stratified split %.0f/%.0f/%.0f  rare-class guarantee: min %d per eval set',
             TRAIN_SPLIT*100, VAL_SPLIT*100, TEST_SPLIT*100, MIN_EVAL_SAMPLES)
    log.info('Rare-class threshold: aspect-class with < %d total -> reserved', rare_threshold)

    # Phase 1: Identify & extract rare-class rows
    rare_indices = set()
    rare_classes = {}

    for aspect in ASPECT_COLUMNS:
        if aspect not in df.columns:
            continue
        vc = df[aspect].value_counts(dropna=True)
        for label, count in vc.items():
            if count < rare_threshold and count >= 3:
                mask    = df[aspect] == label
                indices = df.index[mask].tolist()
                rare_classes[(aspect, label)] = indices
                rare_indices.update(indices)
                log.info('  Rare class: %s=%s (%d samples) — reserved', aspect, label, count)

    rare_df     = df.loc[sorted(rare_indices)].copy()
    majority_df = df.drop(index=sorted(rare_indices)).copy()
    log.info('Reserving %d rare rows  |  %d rows for normal split', len(rare_df), len(majority_df))

    if len(rare_df) > 0:
        rare_df      = rare_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
        n_rare       = len(rare_df)
        n_val_rare   = max(1, int(n_rare * 0.30))
        n_test_rare  = max(1, int(n_rare * 0.30))
        n_train_rare = n_rare - n_val_rare - n_test_rare
        train_rare   = rare_df.iloc[:n_train_rare]
        val_rare     = rare_df.iloc[n_train_rare : n_train_rare + n_val_rare]
        test_rare    = rare_df.iloc[n_train_rare + n_val_rare :]
        log.info('Rare split — Train: %d  Val: %d  Test: %d',
                 len(train_rare), len(val_rare), len(test_rare))
    else:
        empty = pd.DataFrame(columns=df.columns)
        train_rare = val_rare = test_rare = empty

    # Phase 2: Normal stratified split on majority rows
    majority_df = create_multi_label_target(majority_df)
    train_val, test_majority = train_test_split(
        majority_df, test_size=TEST_SPLIT,
        stratify=majority_df['_stratify_key'], random_state=RANDOM_SEED
    )
    val_size_adj = VAL_SPLIT / (TRAIN_SPLIT + VAL_SPLIT)
    train_majority, val_majority = train_test_split(
        train_val, test_size=val_size_adj,
        stratify=train_val['_stratify_key'], random_state=RANDOM_SEED
    )
    for part in tqdm([train_majority, val_majority, test_majority], desc='Cleaning keys'):
        part.drop(columns=['_stratify_key'], inplace=True, errors='ignore')

    # Merge & shuffle
    train_df = pd.concat([train_majority, train_rare], ignore_index=True).sample(
        frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    val_df   = pd.concat([val_majority,   val_rare],   ignore_index=True).sample(
        frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    test_df  = pd.concat([test_majority,  test_rare],  ignore_index=True).sample(
        frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    n = len(df)
    log.info('Split — Train: %d (%.1f%%)  Val: %d (%.1f%%)  Test: %d (%.1f%%)',
             len(train_df), len(train_df)/n*100,
             len(val_df),   len(val_df)/n*100,
             len(test_df),  len(test_df)/n*100)

    log.info('Rare-class counts in eval sets:')
    for (aspect, label), indices in rare_classes.items():
        log.info('    %s=%s  ->  Val: %d  Test: %d', aspect, label,
                 int((val_df[aspect] == label).sum()),
                 int((test_df[aspect] == label).sum()))

    return train_df, val_df, test_df

print(f"\U0001f552 Done in {time.time() - _start_time:.2f}s")
print("\u2705 Completed: Defining function perform_stratified_split.")

⏳ Starting: Defining function perform_stratified_split...
🕒 Done in 0.00s
✅ Completed: Defining function perform_stratified_split.


## Distribution Analysis & Reporting

After splitting, these functions analyse per-aspect class counts across all three sets, flag rare classes below the 10% threshold, and generate a markdown report + JSON distribution file.

In [6]:
print("\u23f3 Starting: Defining distribution analysis functions...")
import time
_start_time = time.time()

def analyze_class_distribution(df: pd.DataFrame, dataset_name: str = 'Full') -> dict:
    """Return per-aspect class counts and percentages."""
    log.info('Class distribution — %s dataset', dataset_name)
    total = len(df)
    distribution = {}
    for aspect in ASPECT_COLUMNS:
        if aspect not in df.columns:
            continue
        vc = df[aspect].value_counts(dropna=False)
        distribution[aspect] = {
            str(label): {'count': int(cnt), 'percentage': float(cnt / total * 100)}
            for label, cnt in vc.items()
        }
    return distribution


def identify_imbalanced_classes(train_df, val_df, test_df, threshold: float = 10.0) -> dict:
    """Flag aspect-classes below threshold percent in the training set."""
    log.info('Identifying imbalanced classes (threshold < %.1f%%) ...', threshold)
    info = {'threshold_percentage': threshold, 'rare_classes': {}, 'recommendations': []}
    for aspect in ASPECT_COLUMNS:
        if aspect not in train_df.columns:
            continue
        vc    = train_df[aspect].value_counts(dropna=True)
        total = len(train_df)
        rare  = [
            {'label': str(lbl), 'count_train': int(cnt), 'percentage_train': float(cnt/total*100)}
            for lbl, cnt in vc.items() if cnt / total * 100 < threshold
        ]
        if rare:
            info['rare_classes'][aspect] = rare
            log.warning('  %s — rare classes: %s', aspect.upper(), [r['label'] for r in rare])
    return info


def create_class_imbalance_report(train_dist, val_dist, test_dist, imbalance_info) -> str:
    """Generate a markdown summary of class distribution and imbalance."""
    lines = [
        "# Class Imbalance Analysis\n\n",
        f"**Date**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n",
        "## Cleaning Pipeline Applied\n\n",
        "| Stage | Technique | Purpose |\n",
        "|-------|-----------|---------|\n",
        "| 1 | Unicode NFC normalisation | Unify combining characters |\n",
        "| 2 | HTML tag & entity removal | Strip `<br>`, `&amp;` etc. |\n",
        "| 3 | URL / e-mail removal | Remove non-sentiment content |\n",
        "| 4 | Translation artifact normalisation | Fix vi->en filler & punctuation |\n",
        "| 5 | Whitespace collapse | Clean token boundaries |\n\n",
        "## Dataset Split\n\n",
        "| Set | Samples |\n|-----|---------|\n",
        f"| Train | {train_dist.get('_total', 'N/A')} |\n",
        f"| Validation | {val_dist.get('_total', 'N/A')} |\n",
        f"| Test | {test_dist.get('_total', 'N/A')} |\n\n",
        "## Aspect-wise Class Distribution\n",
    ]
    for aspect in ASPECT_COLUMNS:
        if aspect not in train_dist:
            continue
        lines.append(f"\n### {aspect.upper()}\n\n")
        lines.append("| Class | Train | Val | Test |\n|-------|-------|-----|------|\n")
        all_labels = (set(train_dist.get(aspect, {}).keys()) |
                      set(val_dist.get(aspect,   {}).keys()) |
                      set(test_dist.get(aspect,  {}).keys()))
        for label in sorted(all_labels):
            def _fmt(dist, a, lbl):
                d = dist.get(a, {}).get(lbl, {'count': 0, 'percentage': 0.0})
                return f"{d['count']} ({d['percentage']:.1f}%)"
            lines.append(f"| {label} | {_fmt(train_dist, aspect, label)} |"
                         f" {_fmt(val_dist, aspect, label)} | {_fmt(test_dist, aspect, label)} |\n")
    if imbalance_info["rare_classes"]:
        lines.append("\n## Imbalanced Classes\n\n")
        lines.append(f"Threshold: < {imbalance_info['threshold_percentage']}% in training set\n\n")
        for aspect, rare_list in imbalance_info["rare_classes"].items():
            lines.append(f"### {aspect.upper()}\n")
            for rc in rare_list:
                lines.append(f"- **{rc['label']}**: {rc['count_train']} samples ({rc['percentage_train']:.2f}%)\n")
    else:
        lines.append("\n*No severely imbalanced classes detected.*\n")
    return "".join(lines)


def save_splits(train_df, val_df, test_df):
    """Save the three split CSVs to OUTPUT_DIR."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for name, split in [('train', train_df), ('val', val_df), ('test', test_df)]:
        path = os.path.join(OUTPUT_DIR, f"{name}.csv")
        split.to_csv(path, index=False)
        log.info('Saved %s -> %s (%d rows)', name, path, len(split))

print(f"\U0001f552 Done in {time.time() - _start_time:.2f}s")
print("\u2705 Completed: Defining distribution analysis functions.")

⏳ Starting: Defining distribution analysis functions...
🕒 Done in 0.00s
✅ Completed: Defining distribution analysis functions.


## Run Pipeline

Executes the full preprocessing and splitting pipeline end to end.

In [7]:
print("\u23f3 Starting: Running preprocessing pipeline...")
import time
_start_time = time.time()

def main():
    log.info('=' * 65)
    log.info('Data Preprocessing and Stratified Splitting')
    log.info('=' * 65)

    # Step 1: Load
    df = load_and_explore_data()
    df_raw = df.copy()

    # Step 2: Clean text
    df = apply_cleaning_pipeline(df)
    log_cleaning_examples(df_raw, df)

    # Step 3: Drop rows with empty text after cleaning
    before = len(df)
    df = df.dropna(subset=['data'])
    log.info('Dropped %d empty-text rows (%d remaining)', before - len(df), len(df))

    # Step 4: Drop rows with no aspect labels at all
    before = len(df)
    df = df.dropna(subset=ASPECT_COLUMNS, how='all')
    log.info('Dropped %d all-NaN-label rows (%d remaining)', before - len(df), len(df))

    # Step 5: Two-phase stratified split
    train_df, val_df, test_df = perform_stratified_split(df)

    # Step 6: Distribution analysis
    train_dist = analyze_class_distribution(train_df, 'Training')
    val_dist   = analyze_class_distribution(val_df,   'Validation')
    test_dist  = analyze_class_distribution(test_df,  'Test')
    train_dist['_total'] = len(train_df)
    val_dist['_total']   = len(val_df)
    test_dist['_total']  = len(test_df)

    # Step 7: Imbalance detection
    imbalance_info = identify_imbalanced_classes(train_df, val_df, test_df)

    # Step 8: Save CSVs
    save_splits(train_df, val_df, test_df)

    # Step 9: Markdown report
    report = create_class_imbalance_report(train_dist, val_dist, test_dist, imbalance_info)
    report_path = os.path.join(os.path.dirname(OUTPUT_DIR), 'class_imbalance.md')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report)
    log.info('Saved class imbalance report -> %s', report_path)

    # Step 10: JSON distribution dump
    dist_json = {'train': train_dist, 'val': val_dist, 'test': test_dist,
                 'imbalance_info': imbalance_info}
    json_path = os.path.join(OUTPUT_DIR, 'class_distribution.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(dist_json, f, indent=2)
    log.info('Saved distribution data -> %s', json_path)

    log.info('=' * 65)
    log.info('PREPROCESSING COMPLETE')
    log.info('=' * 65)


main()

print(f"\U0001f552 Done in {time.time() - _start_time:.2f}s")
print("\u2705 Completed: Running preprocessing pipeline.")

2026-03-20 21:04:19 [INFO] =================================================================
2026-03-20 21:04:19 [INFO] Data Preprocessing and Stratified Splitting
2026-03-20 21:04:19 [INFO] =================================================================
2026-03-20 21:04:19 [INFO] Loading data from c:\Users\lucif\Desktop\Clearview\ml-research\data\raw\full_data_en.csv ...
2026-03-20 21:04:19 [INFO] Loaded 13241 rows x 8 columns
2026-03-20 21:04:19 [INFO] Columns: ['data', 'stayingpower', 'texture', 'smell', 'price', 'colour', 'shipping', 'packing']
2026-03-20 21:04:19 [INFO] Missing values:
data                0
stayingpower    10465
texture          8362
smell           10336
price            9970
colour           5744
shipping         7816
packing         10235
2026-03-20 21:04:19 [INFO] Applying cleaning pipeline to: ['data']


⏳ Starting: Running preprocessing pipeline...


100%|██████████| 13241/13241 [00:00<00:00, 20393.75it/s]
2026-03-20 21:04:20 [INFO]   data             newly emptied:    1  |  total missing:    1
2026-03-20 21:04:20 [INFO] === Cleaning examples (column: data) ===
2026-03-20 21:04:20 [INFO] Dropped 1 empty-text rows (13240 remaining)
2026-03-20 21:04:20 [INFO] Dropped 0 all-NaN-label rows (13240 remaining)
2026-03-20 21:04:20 [INFO] Stratified split 70/15/15  rare-class guarantee: min 8 per eval set
2026-03-20 21:04:20 [INFO] Rare-class threshold: aspect-class with < 53 total -> reserved
2026-03-20 21:04:20 [INFO]   Rare class: price=neutral (26 samples) — reserved
2026-03-20 21:04:20 [INFO]   Rare class: price=negative (21 samples) — reserved
2026-03-20 21:04:20 [INFO]   Rare class: packing=neutral (18 samples) — reserved
2026-03-20 21:04:20 [INFO] Reserving 65 rare rows  |  13175 rows for normal split
2026-03-20 21:04:20 [INFO] Rare split — Train: 27  Val: 19  Test: 19
2026-03-20 21:04:20 [INFO] Building stratification key ...



  [BEFORE] The color is quite similar to the lipstick picture, but I like more pink than buying 203
  [AFTER ] The color is quite similar to the lipstick picture, but I like more pink than buying 203

  [BEFORE] This item was ordered on January 2nd due to border congestion so it arrived a bit late but still ok
  [AFTER ] This item was ordered on January 2nd due to border congestion so it arrived a bit late but still ok

  [BEFORE] Dries quickly, is super waterproof, I scrubbed vigorously under the tap but nothing happened!  More than the expensive kind
  [AFTER ] Dries quickly, is super waterproof, I scrubbed vigorously under the tap but nothing happened! More than the expensive kind

  [BEFORE] Uses: Good
Texture: Good
Color fastness: Good ??
The lipstick has a beautiful color and is so easy to admire??hhhhhhhhhhhh
  [AFTER ] Uses: Good Texture: Good Color fastness: Good ? The lipstick has a beautiful color and is so easy to admire?hhhhhhhhhhhh

  [BEFORE] The lipstick is so beautifu

2026-03-20 21:04:20 [INFO] Rare stratification keys (n=1): 393 — grouped as other_rare_combination


Cleaning keys:   0%|          | 0/3 [00:00<?, ?it/s]

2026-03-20 21:04:20 [INFO] Split — Train: 9248 (69.8%)  Val: 1996 (15.1%)  Test: 1996 (15.1%)
2026-03-20 21:04:20 [INFO] Rare-class counts in eval sets:
2026-03-20 21:04:20 [INFO]     price=neutral  ->  Val: 6  Test: 11
2026-03-20 21:04:20 [INFO]     price=negative  ->  Val: 7  Test: 5
2026-03-20 21:04:20 [INFO]     packing=neutral  ->  Val: 6  Test: 3
2026-03-20 21:04:20 [INFO] Class distribution — Training dataset
2026-03-20 21:04:20 [INFO] Class distribution — Validation dataset
2026-03-20 21:04:20 [INFO] Class distribution — Test dataset
2026-03-20 21:04:20 [INFO] Identifying imbalanced classes (threshold < 10.0%) ...
2026-03-20 21:04:20 [WARNING]   STAYINGPOWER — rare classes: ['negative', 'neutral']
2026-03-20 21:04:20 [WARNING]   TEXTURE — rare classes: ['negative', 'neutral']
2026-03-20 21:04:20 [WARNING]   SMELL — rare classes: ['negative', 'neutral']
2026-03-20 21:04:20 [WARNING]   PRICE — rare classes: ['neutral', 'negative']
2026-03-20 21:04:20 [WARNING]   COLOUR — rare cla

🕒 Done in 1.21s
✅ Completed: Running preprocessing pipeline.
